# Pixel-level Marker Binarization & Spatial ECM Analysis

This notebook processes multiplex immunofluorescence pixel-level data (Lunaphore at 10 µm resolution) to quantify ECM remodeling and epithelial changes across H&E-annotated tissue regions at **Day 1** vs **Day 60** post-treatment.

**Pipeline overview:**
1. Load pixel AnnData and metadata
2. Transfer H&E region annotations (GLANDS, ECM, FAT) to pixels
3. Compute acini distance and spatial bin assignments
4. **Marker binarization** using `final_smart_threshold` (GMM-based, per region)
5. Area fraction analysis: ECM and epithelial markers across spatial bins
7. ECM island–level collagen clustering (clustermap per ECM region)
8. Tissue composition and proportion summaries

---

**Binarization method — `final_smart_threshold`** (the final method used):

BIC model selection chooses between 1-, 2-, or 3-component GMMs per marker per region:
- **Bimodal valley** (2-component, well-separated): threshold at the intersection of the two Gaussians
- **Broad bimodal shoulder** (2-component, overlapping): threshold at shoulder of the dominant peak
- **Trimodal upper valley** (3-component): threshold between mid and high components
- **Unimodal shoulder** (1-component): threshold at `mean − 1.5σ`

Thresholding is run **separately per H&E region** (ECM, GLANDS, FAT) within each section (`cid`), then combined into the final layer `binary_perregion`. FAT pixels from `binary_perregion` are then merged into the no-FAT-background layer `binary_nofat`, which is the layer used for all downstream analyses.

**Output layers in `adata`:**
| Layer | Description |
|-------|-------------|
| `raw_convolved` | Raw pixel intensities (Gaussian convolved) |
| `binary_perregion` | Binary mask per marker, thresholded per H&E region |
| `binary_nofat` | Same as above but with FAT-region pixels merged in |

**Tissue region labels (`annot_he`):** `GLANDS`, `ECM`, `FAT`, `background`  
**Spatial bins (`spatial_bin`):** `intra-acini`, `bin_4cells` … `bin_40cells`, `inf` (distance from nearest acinus in cell-diameter units, d_cell ≈ 4.37 µm)

## Imports

In [ ]:
import gc
import tqdm
import tifffile
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter1d
from skimage import measure
from skimage.color import label2rgb
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from statannot import add_stat_annotation

import scanpy as sc
import anndata as ad

mpl.rcParams['pdf.fonttype'] = 42

## Configuration

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
ADATA_CELL_PATH   = '../data/adata_merged_asinh10_combat.h5ad'
ADATA_PX_PATH     = '../data/pixel_anndata_naive_10um.h5ad'
ALIGNED_MASK_DIR  = '../data/aligned_tissuemask/'
CROP_IDX_PATH     = '../data/tissue_crop_idx_dapi_lv4_basedonmask.csv'
IMGPATH_CSV       = '../data/imgpath_metadata.csv'

RESULTS_DIR       = '../results/'
FIGS_DIR          = '../figures/'
FIGS_ECM_DIR      = '../figures/ecm_cols/'
SAVEPTH_BIN_QC    = '../figures/binary_qc_perregion/'
for d in [RESULTS_DIR, FIGS_DIR, FIGS_ECM_DIR, SAVEPTH_BIN_QC]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ── Marker metadata ────────────────────────────────────────────────────────────
DICT_MARKERS = {
    "CD11B": "Myeloid (mac/DC)", "CD11C": "Myeloid (mac/DC)",
    "CD138": "Plasma cell", "CD146": "Pericyte / vascular-associated",
    "CD20": "B cell", "CD3": "T cell", "CD31": "Endothelial (blood vessel)",
    "CD4": "CD4 T cell", "CD44": "Tumor/activated (broad)", "CD45": "Immune",
    "CD56": "NK cell", "CD7": "T/NK cell", "CD8": "CD8 T cell",
    "CK5": "Basal epithelial", "CK7": "Luminal epithelial",
    "COL10A1": "ECM / CAF-like (remodeling)", "COL6A1": "ECM / fibroblast",
    "COLT1": "ECM (collagen-related)", "COLT2": "ECM (collagen-related)",
    "COLT3": "ECM (collagen-related)", "COLT4": "ECM (collagen-related)",
    "COLT5A1": "ECM (collagen-related)",
    "ECADHERIN": "Epithelial", "ERA": "ER+ luminal epithelial",
    "FAP": "CAF (activated fibroblast)", "FBLN1": "ECM / fibroblast",
    "FIBRONECTIN": "ECM / fibroblast", "HAPLN1": "ECM / fibroblast",
    "KI67": "Proliferating", "LYVE1": "Lymphatic endothelium",
    "MMP9": "Myeloid (inflammatory)", "NCADHERIN": "Mesenchymal / EMT-like",
    "PANCK": "Epithelial", "PAX5": "B cell", "PDGFRb": "Pericyte",
    "PDPN": "Fibroblast / CAF-like", "PGRMC1": "ER+ luminal epithelial",
    "PLIN1": "Adipocyte", "VIMENTIN": "Mesenchymal (stromal/EMT)",
    "aSMA": "Myofibroblast / pericyte",
}

# ── Sample metadata ────────────────────────────────────────────────────────────
DICT_CID2PID = {
    'slide1_1622753': 4,  'slide4_1629501': 6,   'slide4_1629507': 10,
    'slide5_1622930': 6,  'slide2_1621945': 7,   'slide2_1622668': 3,
    'slide1_top_1623464': 2, 'slide3_topmiddle_1615778': 2,
    'slide3_topmiddle_1615782': 3, 'slide4_bottom_1631298': 9,
    'slide4_bottom_1629501': 6,    'slide3_bottom_1623790': 5,
    'slide2_top_1614806': 4,
}

# Sections with anomalous boundary pixels — excluded from paired analysis
RMCIDS = ['slide1_1622753', 'slide4_1629501', 'slide4_1629507', 'slide4_bottom_1631298']

# ── H&E region mapping ─────────────────────────────────────────────────────────
ANNOT_LIST    = ['FAT', 'ECM', 'GLANDS']
DICT_REGION2NUM = {'GLANDS': 1, 'ECM': 2, 'FAT': 3}
DICT_NUM2REGION = {0: 'background', 1: 'GLANDS', 2: 'ECM', 3: 'FAT'}

# ── Spatial binning ────────────────────────────────────────────────────────────
D_CELL        = 4.371621916951665  # µm per cell diameter at 10 µm resolution
SCALE_FACTOR  = 4.8               # µm per pixel at level 4

# ── Plot aesthetics ────────────────────────────────────────────────────────────
DICT_COLOR    = {'Day1': '#A1BE96', 'Day60': '#F58867'}
HUE_ORDER     = ['Day1', 'Day60']
DICT_PIDCOLOR = {
    7: (0.40, 0.76, 0.65), 4: (0.99, 0.55, 0.38), 5: (0.55, 0.63, 0.80),
    2: (0.91, 0.54, 0.76), 3: (0.65, 0.85, 0.33), 6: (1.00, 0.85, 0.18),
    10:(0.90, 0.77, 0.58), 9: (0.70, 0.70, 0.70),
}

---
## 1. Load Data

Load the pixel-level AnnData object (10 µm resolution, Gaussian-convolved intensities) and the cell-level AnnData used to derive the timepoint/patient ID lookup.

In [ ]:
# Cell-level object — used only to build the cid → timepoint/patient mapping
adata_cell = sc.read_h5ad(ADATA_CELL_PATH)

df_tid2type = adata_cell.obs[['batch', 'Event_Label']].drop_duplicates().reset_index(drop=True)
dict_tid2type = {
    r.batch.lower().replace('__', '_'): r.Event_Label.replace(' ', '')
    for _, r in df_tid2type.iterrows()
}

# Pixel-level object (10 µm bins, all sections)
adata = sc.read_h5ad(ADATA_PX_PATH)
adata.layers['raw_intensity'] = adata.X.copy()

varnm = list(adata.var_names)
print(adata)

In [ ]:
# Load crop coordinates and image path metadata
df_idx_tissue_dapi = pd.read_csv(CROP_IDX_PATH, index_col=0)
df_path = pd.read_csv(IMGPATH_CSV, index_col=0)

# Add sample metadata to pixels
adata.obs['Type'] = adata.obs.cid.map(dict_tid2type).astype('category')
adata.obs['pid']  = adata.obs.cid.map(DICT_CID2PID)
print(f"Sections: {adata.obs.cid.nunique()}, Pixels: {adata.n_obs:,}")

---
## 2. Transfer H&E Region Annotations

Aligned H&E segmentation masks (GLANDS, ECM, FAT) stored as `.npy` arrays are projected onto pixel coordinates to assign each pixel to a tissue compartment.

In [ ]:
def transfer_he_annotations(adata, df_idx_tissue_dapi, savepth_alignedmask,
                            annot_list, dict_region2num, dict_num2region,
                            show_plots=True):
    """
    Project H&E segmentation masks onto pixel coordinates in adata.
    Adds 'annot_he' column to adata.obs.
    """
    adata.obs['annot_he'] = ''
    for cid in df_idx_tissue_dapi.index:
        img_msk = []
        for a in annot_list:
            nda = np.load(f'{savepth_alignedmask}/{cid}_{a}.npy')
            if len(img_msk) == 0:
                img_msk = nda[:, :, 1].copy()
            img_msk[nda[:, :, 1] != 0] = dict_region2num[a]

        aobs_sub = adata.obs.loc[adata.obs.cid == cid, ['x', 'y']]
        xs = aobs_sub.y.values.astype('int')
        ys = aobs_sub.x.values.astype('int')

        annot_labels = []
        for x, y in zip(xs, ys):
            try:
                annot_labels.append(img_msk[y, x])
            except IndexError:
                annot_labels.append(0)

        aobs_sub['annot_he'] = annot_labels
        adata.obs.loc[aobs_sub.index, 'annot_he'] = (
            aobs_sub['annot_he'].map(dict_num2region)
        )

        if show_plots:
            tmp = pd.DataFrame({'x': xs, 'y': ys, 'annot': pd.Series(annot_labels).map(dict_num2region)})
            f, ax = plt.subplots(figsize=(10, 10))
            plt.imshow(img_msk, cmap='gray_r')
            sns.scatterplot(data=tmp, x='x', y='y', hue='annot', s=0.5, alpha=0.5, ax=ax)
            plt.title(cid)
            plt.show()

    return adata

adata = transfer_he_annotations(
    adata, df_idx_tissue_dapi, ALIGNED_MASK_DIR,
    ANNOT_LIST, DICT_REGION2NUM, DICT_NUM2REGION, show_plots=True
)

---
## 3. Acini Distance & Spatial Bin Assignment

For each pixel, compute the Euclidean distance (µm) to the nearest acinus (GLANDS mask), label the nearest acinus ID, and bin pixels into distance rings.

**Spatial bins** (in cell-diameter units, 1 cell ≈ 4.37 µm):
- `intra-acini` — within the gland lumen (distance = 0)
- `bin_4cells` through `bin_40cells` — periductal stroma in 4-cell increments
- `inf` — beyond 40 cell diameters

Additionally, a coarser grouping `spatial_bin_wide` merges fine bins into `intra-acini`, `bin_8cells`, `bin_20cells`, `bin_40cells`, and `inf`, and `spatial_bin_vsacini` collapses everything to `intra-acini` vs `non-acini`.

In [ ]:
from skimage import measure
from scipy.ndimage import distance_transform_edt

adata.obs[['acini_id', 'spatial_bin', 'dist_to_acini_um']] = None

for cid in df_idx_tissue_dapi.index:
    nda = np.load(f'{ALIGNED_MASK_DIR}/{cid}_GLANDS.npy')
    img_msk = nda[:, :, 1].copy()
    img_msk[nda[:, :, 1] != 0] = 1

    labeled_acini, num_acini = measure.label(img_msk, connectivity=2, return_num=True)
    print(f"{cid}: {num_acini} acini found")

    dist_px, indices = distance_transform_edt(img_msk == 0, return_indices=True)
    dist_um = dist_px * SCALE_FACTOR

    aobs_sub = adata.obs.loc[adata.obs.cid == cid, ['x', 'y']].copy()
    aobs_sub[['x_centroid_px', 'y_centroid_px']] = aobs_sub[['y', 'x']]

    x_px = aobs_sub.x_centroid_px.values.astype('int')
    y_px = aobs_sub.y_centroid_px.values.astype('int')

    dist_um_vector, labeled_acini_vector = [], []
    for x, y in zip(x_px, y_px):
        try:
            dist_um_vector.append(dist_um[y, x])
            nearest_y = indices[0, y, x]
            nearest_x = indices[1, y, x]
            labeled_acini_vector.append(labeled_acini[nearest_y, nearest_x])
        except IndexError:
            dist_um_vector.append(1000)
            labeled_acini_vector.append(-1)

    aobs_sub['dist_to_acini_um'] = dist_um_vector
    aobs_sub['acini_id'] = pd.Categorical(labeled_acini_vector)

    bin_edges = [a * D_CELL for a in [0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40, np.inf]]
    bin_labels = [f'bin_{a}cells' for a in [4, 8, 12, 16, 20, 24, 28, 32, 36, 40]] + ['inf']
    aobs_sub['spatial_bin'] = pd.cut(
        aobs_sub['dist_to_acini_um'], bins=bin_edges, labels=bin_labels, include_lowest=True
    )
    aobs_sub['spatial_bin'] = aobs_sub['spatial_bin'].cat.add_categories(['intra-acini'])
    aobs_sub.loc[aobs_sub['dist_to_acini_um'] == 0, 'spatial_bin'] = 'intra-acini'

    adata.obs.loc[aobs_sub.index, ['acini_id', 'spatial_bin', 'dist_to_acini_um']] = (
        aobs_sub[['acini_id', 'spatial_bin', 'dist_to_acini_um']]
    )

adata.obs.acini_id = adata.obs.acini_id.astype('category')
adata.obs.dist_to_acini_um = adata.obs.dist_to_acini_um.astype('float')

# Coarser spatial groupings
adata.obs['spatial_bin_wide'] = adata.obs['spatial_bin'].astype('str')
for fine, wide in [
    (['bin_40cells','bin_36cells','bin_32cells','bin_28cells','bin_24cells'], 'bin_40cells'),
    (['bin_20cells','bin_16cells','bin_12cells'], 'bin_20cells'),
    (['bin_8cells','bin_4cells'], 'bin_8cells'),
]:
    adata.obs.loc[adata.obs.spatial_bin.isin(fine), 'spatial_bin_wide'] = wide
adata.obs['spatial_bin_wide'] = adata.obs['spatial_bin_wide'].astype('category')

adata.obs['spatial_bin_vsacini'] = 'non-acini'
adata.obs.loc[adata.obs.spatial_bin == 'intra-acini', 'spatial_bin_vsacini'] = 'intra-acini'

print("Spatial bins:", adata.obs.spatial_bin_wide.unique().tolist())

In [ ]:
# Remove anomalous boundary pixels in slide3_topmiddle_1615782 (x > 650 are out-of-tissue)
cid = 'slide3_topmiddle_1615782'
rmids = adata.obs[(adata.obs.cid == cid) & (adata.obs.x > 650)].index
adata = adata[~adata.obs.index.isin(rmids)].copy()
print(f"Removed {len(rmids)} boundary pixels from {cid}. Remaining: {adata.n_obs:,}")

# Normalize and save checkpoint
adata.write(f'{RESULTS_DIR}adata_px_binary_heannot_naive_convoluted_10um_cleaned.h5ad')

---
## 4. Marker Binarization — `final_smart_threshold`

This is the **final binarization method** used in all downstream analyses. See the pipeline overview at the top for the full rationale and comparison with earlier approaches.

**Key design decisions:**
- BIC model selection (1 vs 2 vs 3 components) avoids forcing a bimodal assumption
- Per-region thresholding (separately for ECM, GLANDS, FAT within each section) prevents adipose tissue background from inflating ECM thresholds
- FAT pixels are thresholded by the `binary_perregion` pipeline and merged into `binary_nofat` (which uses non-FAT pixels only for threshold estimation)

In [ ]:
def asinh_transform(data, cofactor=10):
    """arcsinh transformation with cofactor normalization."""
    return np.arcsinh(data / cofactor)

def gaussian_pdf(x, m, s, w):
    return (w / (s * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - m) / s) ** 2)

def classify_unimodal_population(data_transformed, background_cutoff=None):
    """
    For a single-peak distribution, infer whether it is predominantly
    positive or negative signal.
    """
    model = GaussianMixture(n_components=1, random_state=42)
    model.fit(data_transformed.reshape(-1, 1))
    peak_mean = model.means_[0][0]
    if background_cutoff is None:
        status = "Likely Negative" if peak_mean < 2.0 else "Likely Positive"
    else:
        status = "All Positive" if peak_mean > background_cutoff else "All Negative"
    return status, peak_mean

def final_smart_threshold(data_transformed, bins=128, needplot=False, verbose=False):
    """
    Adaptive GMM-based thresholding using BIC model selection.

    Fits 1-, 2-, and 3-component GMMs and selects the best via BIC.
    Threshold strategy depends on selected model:
      - 1-component (unimodal):    threshold at mean − 1.5σ
      - 2-component (bimodal):     intersection valley (well-separated)
                                   or shoulder of dominant peak (overlapping)
      - 3-component (trimodal):    valley between mid and high components

    Parameters
    ----------
    data_transformed : 1D np.ndarray
        asinh-transformed pixel intensities.
    bins : int
        Number of histogram bins for visualization.
    needplot : bool
        If True, also returns histogram and GMM curve data.
    verbose : bool
        If True, prints the selected model and threshold.

    Returns
    -------
    threshold : float or None
        Decision threshold in asinh space.
    status : str
        One of 'Shoulder_Unimodal', 'Bimodal_Valley', 'Shoulder_Broad_Bimodal',
        'Trimodal_Upper_Valley', 'Low_Signal'.
    (+ histogram/curve data if needplot=True)
    """
    vals = data_transformed.flatten()
    vals = vals[vals > 0.1].reshape(-1, 1)

    if len(vals) < 50:
        if not needplot:
            return None, "Low_Signal"
        return None, None, None, None, None, None, None, "Low_Signal"

    counts, bin_edges = np.histogram(data_transformed, bins=bins)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    smoothed = gaussian_filter1d(counts, sigma=3)

    gmm1 = GaussianMixture(n_components=1, random_state=42).fit(vals)
    gmm2 = GaussianMixture(n_components=2, random_state=42).fit(vals)
    gmm3 = GaussianMixture(n_components=3, random_state=42).fit(vals)
    best_n = np.argmin([gmm1.bic(vals), gmm2.bic(vals), gmm3.bic(vals)]) + 1

    if best_n == 1:
        peak_mean = gmm1.means_[0][0]
        peak_std  = np.sqrt(gmm1.covariances_[0][0])
        threshold = peak_mean - (1.5 * peak_std)
        status    = "Shoulder_Unimodal"
        means, stds, weights = gmm1.means_.flatten(), np.sqrt(gmm1.covariances_.flatten()), gmm1.weights_.flatten()
        x_range = np.linspace(means[0] - 3*stds[0], means[0] + 3*stds[0], 500)
        pdf0    = gaussian_pdf(x_range, means[0], stds[0], weights[0])
        pdf1    = None

    elif best_n == 2:
        means   = gmm2.means_.flatten()
        stds    = np.sqrt(gmm2.covariances_.flatten())
        weights = gmm2.weights_.flatten()
        sep = abs(means[0] - means[1]) / (2 * (stds[0] + stds[1]))
        high_idx, low_idx = np.argmax(means), np.argmin(means)
        if sep < 0.8:
            threshold = np.max(means) - (1.2 * stds[high_idx])
            status    = "Shoulder_Broad_Bimodal"
        else:
            x_range = np.linspace(means[low_idx], means[high_idx], 500)
            p_low   = gaussian_pdf(x_range, means[low_idx],  stds[low_idx],  weights[low_idx])
            p_high  = gaussian_pdf(x_range, means[high_idx], stds[high_idx], weights[high_idx])
            threshold = x_range[np.argmin(np.abs(p_low - p_high))]
            status    = "Bimodal_Valley"
        x_range = np.linspace(min(means), max(means), 500)
        pdf0 = gaussian_pdf(x_range, means[0], stds[0], weights[0])
        pdf1 = gaussian_pdf(x_range, means[1], stds[1], weights[1])

    else:  # 3 components
        means   = gmm3.means_.flatten()
        stds    = np.sqrt(gmm3.covariances_.flatten())
        weights = gmm3.weights_.flatten()
        sorted_idx = np.argsort(means)
        mid_idx, high_idx = sorted_idx[1], sorted_idx[2]
        x_range = np.linspace(means[mid_idx], means[high_idx], 500)
        p_mid   = gaussian_pdf(x_range, means[mid_idx],  stds[mid_idx],  weights[mid_idx])
        p_high  = gaussian_pdf(x_range, means[high_idx], stds[high_idx], weights[high_idx])
        threshold = x_range[np.argmin(np.abs(p_mid - p_high))]
        status    = "Trimodal_Upper_Valley"
        pdf0 = gaussian_pdf(x_range, means[0], stds[0], weights[0])
        pdf1 = gaussian_pdf(x_range, means[1], stds[1], weights[1])

    if verbose:
        print(f"  [{status}] threshold = {threshold:.3f}")

    if not needplot:
        return threshold, status
    return threshold, counts, bin_centers, smoothed, x_range, pdf0, pdf1, status

In [ ]:
# ── Run binarization per region (ECM, GLANDS, FAT) per section ──────────────
# This loop produces 'binary_perregion'.
# FAT pixels are thresholded separately per their own distribution;
# non-FAT pixels are thresholded on the ECM+GLANDS distribution combined.

LAYERNM_PER_REGION = 'binary_perregion'
TARGET_MARKERS = [m for m in varnm if m in DICT_MARKERS]

adata.layers[LAYERNM_PER_REGION] = np.zeros(adata.layers['raw_convolved'].shape, dtype=np.uint8)

for cid in tqdm.tqdm(adata.obs.cid.unique()):
    sid = '_'.join(cid.split('_')[:-1])
    f_he = df_path.loc[sid].hepath
    with tifffile.TiffFile(f_he) as tif:
        image_he = tif.series[0].levels[4].asarray()
    imgidx = df_idx_tissue_dapi.loc[cid]

    cid_mask    = adata.obs.cid == cid
    cid_indices = np.where(cid_mask)[0]
    asub        = adata[cid_indices].copy()

    for marker in TARGET_MARKERS:
        idx       = varnm.index(marker)
        raw_img   = asub.layers['raw_convolved'][:, idx]
        transformed_data = asinh_transform(raw_img)

        cid_full_binary = np.zeros(len(cid_indices), dtype=np.uint8)

        for region in asub.obs.annot_he.unique():
            region_mask_in_sub  = (asub.obs.annot_he == region)
            transformed_sub     = transformed_data[region_mask_in_sub]

            tmp = asub.obs.loc[region_mask_in_sub, ['x', 'y']].copy()
            tmp['asinh'] = transformed_sub

            thresh, counts, centers, smoothed, x_range, pdf0, pdf1, status = (
                final_smart_threshold(transformed_sub, needplot=True)
            )

            if thresh is not None:
                binary_mask = (transformed_sub > thresh).astype(np.uint8)
                cid_full_binary[region_mask_in_sub] = binary_mask
                tmp[LAYERNM_PER_REGION] = binary_mask
            else:
                tmp[LAYERNM_PER_REGION] = 0

            # QC plot: H&E | asinh image | histogram | binary mask
            fig, axs = plt.subplots(2, 2, figsize=(14, 10))
            axr = axs.ravel()
            axr[0].imshow(image_he[imgidx.y0:imgidx.yf, imgidx.x0:imgidx.xf, :])
            axr[0].invert_xaxis(); axr[0].set_title('H&E')
            axr[1].imshow(df2img(tmp, 'asinh'), cmap='magma')
            axr[1].set_title(f'{marker} (asinh)'); axr[1].axis('off')
            axr[2].hist(transformed_sub, bins=128, density=True, color='lightgray', alpha=0.5)
            if thresh is not None:
                axr[2].axvline(thresh, color='blue', lw=2, label=status)
            axr[2].set_title('Histogram & Threshold'); axr[2].legend(fontsize='small')
            if thresh is not None:
                axr[3].imshow(df2img(tmp, LAYERNM_PER_REGION), cmap='gray_r')
                axr[3].set_title('Binary Mask')
            else:
                axr[3].text(0.5, 0.5, 'Unimodal / No Signal', ha='center')
            axr[3].axis('off')
            plt.suptitle(f'{cid} | {marker} | region: {region}')
            plt.tight_layout()
            plt.savefig(f'{SAVEPTH_BIN_QC}/QC_{marker}_{cid}_{region}.png')
            plt.close()

        adata.layers[LAYERNM_PER_REGION][cid_indices, idx] = cid_full_binary

print(f'Binarization complete → adata.layers["{LAYERNM_PER_REGION}"]')

In [ ]:
# ── Build binary_nofat: threshold on non-FAT pixels, then copy FAT from per-region ──
# This avoids adipose autofluorescence biasing ECM/GLANDS thresholds.
LAYERNM_NOFAT = 'binary_nofat'
adata.layers[LAYERNM_NOFAT] = np.zeros(adata.layers['raw_convolved'].shape, dtype=np.uint8)

for cid in tqdm.tqdm(adata.obs.cid.unique()):
    sid = '_'.join(cid.split('_')[:-1])
    f_he = df_path.loc[sid].hepath
    with tifffile.TiffFile(f_he) as tif:
        image_he = tif.series[0].levels[4].asarray()
    imgidx = df_idx_tissue_dapi.loc[cid]

    cid_indices = np.where(adata.obs.cid == cid)[0]
    asub        = adata[cid_indices].copy()

    for marker in TARGET_MARKERS:
        idx      = varnm.index(marker)
        raw_img  = asub.layers['raw_convolved'][:, idx]
        transformed_data = asinh_transform(raw_img)

        # Threshold on non-FAT pixels only
        nonfat_mask = (asub.obs.annot_he != 'FAT')
        transformed_sub = transformed_data[nonfat_mask]

        cid_full_binary = np.zeros(len(cid_indices), dtype=np.uint8)

        thresh, status = final_smart_threshold(transformed_sub, needplot=False)
        if thresh is not None:
            cid_full_binary[nonfat_mask] = (transformed_sub > thresh).astype(np.uint8)

        adata.layers[LAYERNM_NOFAT][cid_indices, idx] = cid_full_binary

# Overwrite FAT pixels with the per-region FAT thresholds
fat_indices = np.where(adata.obs.annot_he == 'FAT')[0]
adata.layers[LAYERNM_NOFAT][fat_indices, :] = (
    adata.layers[LAYERNM_PER_REGION][fat_indices, :].copy()
)
print(f'Layer "{LAYERNM_NOFAT}" ready (non-FAT thresholded + FAT from per-region)')

adata.write(f'{RESULTS_DIR}adata_px_binary_heannot_naive_convoluted_10um_cleaned_final.h5ad')

---
## 5. Helper Functions for Area Fraction Analysis

In [ ]:
def df2img(df, colnm):
    """Convert a pixel DataFrame with x/y coords to a 2D image array."""
    xmax = df.x.max() + 1
    ymax = df.y.max() + 1
    img = np.zeros((ymax, xmax))
    if isinstance(colnm, list):
        img[df.y, df.x] = df[colnm].median(axis=1).astype('int')
    else:
        img[df.y, df.x] = df[colnm].astype('int')
    return img


def compute_area_fractions(adata, layernm, markers, spatial_bin_col,
                            spatial_bins, grouplist, region=None):
    """
    Compute per-marker positive area fraction across spatial bins and patient groups.

    Parameters
    ----------
    adata : AnnData
    layernm : str
        Layer name containing binary (0/1) masks.
    markers : list of str
        Marker names to quantify.
    spatial_bin_col : str
        Column in adata.obs defining spatial bins.
    spatial_bins : list of str
        Bin values to iterate over.
    grouplist : list of str
        Columns to group by (e.g. ['Type', 'cid', 'pid']).
    region : str or None
        If provided, restrict to this H&E region (annot_he == region).

    Returns
    -------
    pd.DataFrame
        Tidy DataFrame with columns [*grouplist, 'area_fraction', 'Marker', 'Bin'].
    """
    all_summaries = []
    for bin_val in spatial_bins:
        for marker in markers:
            m_idx = varnm.index(marker)
            is_region = (adata.obs[spatial_bin_col] == bin_val)
            if region:
                is_region &= (adata.obs['annot_he'] == region)
            is_high = (adata.layers[layernm][:, m_idx] > 0) & is_region

            adata.obs['tmp_region'] = is_region
            adata.obs['tmp_marker'] = is_high

            summary = adata.obs.groupby(grouplist, observed=True).apply(
                lambda x: (x['tmp_marker'].sum() / x['tmp_region'].sum() * 100)
                if x['tmp_region'].sum() > 10 else np.nan,
                include_groups=False
            ).reset_index()
            summary.columns = [*grouplist, 'area_fraction']
            summary['Marker'] = marker
            summary['Bin']    = bin_val
            all_summaries.append(summary)

    return pd.concat(all_summaries, ignore_index=True)


def paired_boxplot(df, x, y, marker_order, hue_order, dict_color, dict_pidcolors,
                   title='', savepath=None, show_annot=True, figsize=(12, 5)):
    """
    Boxplot with paired patient lines and jittered patient-colored dots.
    Assumes df has columns 'pid' and 'Type'.
    """
    df = df.dropna(subset=[y])
    df = df[df[x].isin(marker_order)]

    fig, ax = plt.subplots(figsize=figsize)
    sns.boxplot(x=x, y=y, hue='Type', data=df, order=marker_order,
                palette=dict_color, hue_order=hue_order, ax=ax,
                showfliers=False, boxprops={'alpha': 0.3})

    offsets = {hue_order[0]: -0.2, hue_order[1]: 0.2}
    for i, m in enumerate(marker_order):
        m_data = df[df[x] == m]
        for patient in m_data['pid'].unique():
            p_data  = m_data[m_data['pid'] == patient]
            p_color = dict_pidcolors.get(patient, 'gray')
            for _, row in p_data.iterrows():
                ax.scatter(i + offsets[row['Type']], row[y],
                           color=p_color, s=25, edgecolors='white',
                           linewidth=0.5, zorder=4)
            if len(p_data) == 2:
                y1 = p_data.loc[p_data['Type'] == hue_order[0], y].values[0]
                y2 = p_data.loc[p_data['Type'] == hue_order[1], y].values[0]
                ax.plot([i + offsets[hue_order[0]], i + offsets[hue_order[1]]],
                        [y1, y2], color=p_color, alpha=0.5, linewidth=1, zorder=3)

    if show_annot:
        try:
            pairs = [((g, hue_order[0]), (g, hue_order[1])) for g in marker_order]
            add_stat_annotation(ax, data=df, x=x, y=y, hue='Type',
                                box_pairs=pairs, test='t-test_ind',
                                text_format='star', loc='inside', verbose=0)
        except Exception:
            pass

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], title='Type',
              loc='upper left', bbox_to_anchor=(1, 1))
    plt.xticks(rotation=60, ha='right')
    plt.ylabel('Area fraction (%)')
    plt.xlabel('')
    plt.title(title)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=300)
    plt.show()

---
## 6. ECM Marker Area Fractions across Spatial Bins

Quantify the positive area fraction (%) of ECM, stromal, and adipocyte markers across periductal spatial bins (distance from nearest acinus), comparing Day1 vs Day60.

**Markers:** ECM fibroblast (COL6A1, COL10A1, FIBRONECTIN, HAPLN1, FBLN1), collagen subtypes (COLT1–5A1), CAF activation (FAP, aSMA, PDPN), inflammatory ECM (MMP9, VIMENTIN), and adipocyte (PLIN1).

In [ ]:
LAYERNM      = 'binary_nofat'
SPATIAL_BINS = ['intra-acini', 'bin_8cells', 'bin_20cells']
ECM_MARKERS  = ['COL6A1', 'COL10A1', 'FIBRONECTIN', 'MMP9', 'aSMA', 'FAP',
                 'VIMENTIN', 'PDPN', 'PLIN1', 'COLT1', 'COLT2', 'COLT3', 'COLT4', 'COLT5A1']
MARKER_ORDER = ECM_MARKERS

# Aggregate per patient (grouplist = ['Type', 'cid', 'pid'])
adata.obs['pid'] = adata.obs.cid.map(DICT_CID2PID)
unique_pids      = adata.obs['pid'].dropna().unique()
dict_pidcolors   = dict(zip(unique_pids, sns.color_palette("husl", len(unique_pids))))

df_ecm = compute_area_fractions(
    adata, LAYERNM, ECM_MARKERS,
    spatial_bin_col='spatial_bin',
    spatial_bins=SPATIAL_BINS,
    grouplist=['Type', 'cid', 'pid'],
)

# Plot per spatial bin
for bin_val in SPATIAL_BINS:
    df_sub = df_ecm.query('Bin == @bin_val')
    paired_boxplot(
        df_sub, x='Marker', y='area_fraction', marker_order=MARKER_ORDER,
        hue_order=HUE_ORDER, dict_color=DICT_COLOR, dict_pidcolors=dict_pidcolors,
        title=f'ECM markers — {bin_val}',
        savepath=f'{RESULTS_DIR}boxplot_ECM_{LAYERNM}_{bin_val}_perpatient.pdf',
    )

---
## 8. Tissue Composition and ECM Island Collagen Clustering

### 9a. H&E region proportions per spatial bin
Visualize the composition of GLANDS / ECM / FAT pixels within each spatial bin, per patient paired across timepoints.

### 9b. ECM island–level collagen clustering
Label individual ECM islands (connected components within the ECM mask), assign each pixel to its nearest island, and compute mean per-slide z-scored intensities of collagen markers per island. Clustermap rows = individual ECM islands, colored by patient and timepoint.

In [ ]:
# H&E region proportions within each spatial bin
all_bin_proportions = []
for bin_val in SPATIAL_BINS:
    is_region = (adata.obs['spatial_bin'] == bin_val)
    summary = (
        adata.obs[is_region]
        .groupby(['Type', 'cid', 'pid'], observed=True)['annot_he']
        .value_counts(normalize=True).mul(100)
        .reset_index(name='proportion')
    )
    summary['Bin'] = bin_val
    all_bin_proportions.append(summary)

df_prop = pd.concat(all_bin_proportions, ignore_index=True)
df_prop = df_prop[df_prop.proportion != 0]

# Paired plot: one subplot per H&E region
for bin_val in SPATIAL_BINS:
    df_bin = df_prop.query('Bin == @bin_val')
    he_cats = df_bin['annot_he'].unique()
    fig, axes = plt.subplots(1, len(he_cats), figsize=(5 * len(he_cats), 4))
    if len(he_cats) == 1: axes = [axes]

    offsets = {HUE_ORDER[0]: -0.2, HUE_ORDER[1]: 0.2}
    pid_palette = dict(zip(df_bin['pid'].unique(),
                           sns.color_palette("husl", df_bin['pid'].nunique())))

    for ax, region in zip(axes, he_cats):
        df_sub = df_bin[df_bin['annot_he'] == region]
        sns.boxplot(x='annot_he', y='proportion', hue='Type', data=df_sub,
                    order=[region], hue_order=HUE_ORDER, palette=DICT_COLOR, ax=ax,
                    showfliers=False, boxprops={'alpha': 0.2})
        for patient in df_sub['pid'].unique():
            p_data  = df_sub[df_sub['pid'] == patient]
            p_color = pid_palette.get(patient, 'gray')
            for _, row in p_data.iterrows():
                ax.scatter(0 + offsets[row['Type']], row['proportion'],
                           color=p_color, s=40, edgecolors='white',
                           linewidth=0.8, zorder=5)
            if len(p_data) == 2:
                y1 = p_data.loc[p_data['Type'] == HUE_ORDER[0], 'proportion'].values[0]
                y2 = p_data.loc[p_data['Type'] == HUE_ORDER[1], 'proportion'].values[0]
                ax.plot([offsets[HUE_ORDER[0]], offsets[HUE_ORDER[1]]], [y1, y2],
                        color=p_color, alpha=0.6, linewidth=1.5, zorder=4)
        ax.set_title(region); ax.set_xlabel(''); ax.get_legend().remove()

    plt.suptitle(f'Tissue composition — {bin_val}', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# ECM island labeling and collagen clustermap
MIN_PIXEL_SIZE = 50   # minimum pixels to call an ECM island
COL_MARKERS    = ['COL6A1', 'COL10A1', 'COLT1', 'COLT2', 'COLT3', 'COLT4', 'COLT5A1']

adata.obs[['ecm_region_id']] = None
for cid in df_idx_tissue_dapi.index:
    nda  = np.load(f'{ALIGNED_MASK_DIR}/{cid}_ECM.npy')
    nda2 = np.load(f'{ALIGNED_MASK_DIR}/{cid}_GLANDS.npy')

    img_msk = nda[:, :, 1].copy()
    img_msk[nda[:, :, 1] != 0]  = 1
    img_msk[nda2[:, :, 1] != 0] = 0   # Remove gland overlap from ECM

    labeled_image, _ = measure.label(img_msk, connectivity=2, return_num=True)

    # Keep only ECM islands above minimum size
    dst_labels = np.zeros_like(labeled_image)
    for region in measure.regionprops(labeled_image):
        if region.area >= MIN_PIXEL_SIZE:
            dst_labels[labeled_image == region.label] = region.label

    # Distance transform to assign every pixel to its nearest ECM island
    dist, indices = ndi.distance_transform_edt(dst_labels == 0, return_indices=True)
    nearest_labels = dst_labels[indices[0], indices[1]]

    aobs_sub = adata.obs.loc[adata.obs.cid == cid, ['x', 'y']].copy()
    aobs_sub[['x_centroid_px', 'y_centroid_px']] = aobs_sub[['y', 'x']]
    x_px = aobs_sub.x_centroid_px.values.astype('int')
    y_px = aobs_sub.y_centroid_px.values.astype('int')

    labeled_ecm_vector = []
    for x, y in zip(x_px, y_px):
        try:
            labeled_ecm_vector.append(
                nearest_labels[y, x]
                if 0 <= y < nearest_labels.shape[0] and 0 <= x < nearest_labels.shape[1]
                else 0
            )
        except IndexError:
            labeled_ecm_vector.append(0)

    aobs_sub['ecm_region_id'] = pd.Categorical(labeled_ecm_vector)
    adata.obs.loc[aobs_sub.index, 'ecm_region_id'] = aobs_sub['ecm_region_id']

adata.obs['unique_region_id'] = (
    adata.obs['cid'].astype(str) + '_ecm_' + adata.obs['ecm_region_id'].astype(str)
)
print("ECM island assignment complete.")

In [ ]:
# Per-slide z-score normalization for clustermap
def per_slide_zscore(df):
    return (df - df.mean()) / df.std()

def ecm_island_clustermap(adata, col_markers, spatial_bin=None,
                           layer='raw_convolved', savepath=None):
    """
    Clustermap of collagen marker intensities per ECM island.
    Rows = ECM islands, colored by patient and timepoint.
    """
    mask = adata.obs['ecm_region_id'] > 0
    if spatial_bin:
        mask &= (adata.obs['spatial_bin'] == spatial_bin)
    adata_ecm = adata[mask].copy()

    scaled_X = (
        pd.DataFrame(adata_ecm.layers[layer], index=adata_ecm.obs.index,
                     columns=adata_ecm.var_names)
        .groupby(adata_ecm.obs['cid'])
        .apply(per_slide_zscore)
    )
    adata_ecm.X = scaled_X

    df_int = pd.DataFrame(
        adata_ecm[:, col_markers].X.toarray()
        if hasattr(adata_ecm.X, 'toarray') else adata_ecm[:, col_markers].X,
        index=adata_ecm.obs_names, columns=col_markers
    )
    df_int['unique_region_id'] = adata_ecm.obs['unique_region_id'].values
    region_means = df_int.groupby('unique_region_id').mean()

    region_meta = (
        adata_ecm.obs[['unique_region_id', 'pid', 'Type']]
        .drop_duplicates()
        .set_index('unique_region_id')
        .reindex(region_means.index)
    )
    dict_type_colors = {'Day1': '#FAB31B', 'Day60': '#4766B0'}
    row_colors = pd.DataFrame({
        'Patient ID':  region_meta['pid'].map(dict_pidcolors),
        'Tissue Type': region_meta['Type'].map(dict_type_colors),
    }, index=region_meta.index).astype(object).fillna('white')

    g = sns.clustermap(
        region_means, cmap='RdBu_r', method='ward',
        figsize=(5, max(8, len(region_means) * 0.03)),
        z_score=0, center=0,
        row_colors=row_colors, yticklabels=False,
        cbar_kws={'label': 'Z-score intensity'}
    )
    title = f'ECM collagen — {spatial_bin or "entire tissue"}'
    g.ax_heatmap.set_title(title, pad=20)
    if savepath:
        g.savefig(savepath, dpi=300, bbox_inches='tight')
    plt.show()

for sbin in [None, 'bin_8cells', 'bin_20cells']:
    tag = sbin or 'all'
    ecm_island_clustermap(
        adata, COL_MARKERS, spatial_bin=sbin, layer='raw_convolved',
        savepath=f'{FIGS_ECM_DIR}clustermap_collagen_{tag}_zperslide_zrow.png'
    )

---
## 9. Export Area Tables and Save Final AnnData

In [ ]:
# Area tables for external use (e.g. R-based statistics)
for obs_col, fname in [
    ('annot_he',          'area_entireregion_persample.csv'),
    ('spatial_bin_vsacini','area_spatial_bin_vsacini_persample.csv'),
]:
    cts = adata.obs[['cid', obs_col]].groupby(['cid', obs_col], observed=True).size()
    pd.DataFrame(cts, columns=['area']).to_csv(f'{RESULTS_DIR}{fname}')
    print(f"Saved {fname}")

# Per-gland per-bin area
cts_gland = (
    adata[adata.obs.spatial_bin != 'inf']
    .obs[['cid', 'acini_id', 'spatial_bin', 'annot_he']]
    .groupby(['cid', 'acini_id', 'spatial_bin', 'annot_he'], observed=True)
    .size()
)
pd.DataFrame(cts_gland, columns=['area']).to_csv(
    f'{RESULTS_DIR}area_pergland_perspatialbin_persample.csv'
)

adata.write(f'{RESULTS_DIR}adata_px_binary_heannot_final.h5ad')
print("\nFinal AnnData saved.")
print(adata)